# Deep_GLOC masked-seed benchmark notebook

这个 notebook 用于在 **你已经生成好的 no-leak split bundles** 上批量运行 **Deep_GLOC**，并自动计算 benchmark 指标。

## 会做什么
1. 遍历 `pyg_split_bundles/<split_id>/pyg_data_no_leak.pt`
2. 对每个 split 运行多次 Deep_GLOC 训练
3. 聚合多次训练得到平均概率
4. 对当前 process 进行候选基因排序
5. 计算以下指标：
   - Recall@20
   - Recall@50
   - Recall@100
   - median rank
   - MRR
   - AUROC
   - AUPRC
6. 输出：
   - 每个 split 的 ranked genes
   - 每个 split 的 metrics
   - 每个 process 的平均 metrics
   - 四个 process 的 macro-average

## 关键说明
- 这里默认**在候选基因集合上**评估，也就是 `is_supervised_seed == 0` 的节点参与排序。
- 这意味着：
  - train seeds 不参与排名
  - held-out seeds 作为隐藏真阳性参与评估
  - 其他非监督节点作为候选负类 / 未标记背景
- 这是 masked-seed recovery 最常见、也最公平的评估方式。


In [1]:
import os
import copy
import random
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.nn import GATv2Conv

warnings.filterwarnings("ignore")


In [2]:
# ============================================================
# 1. 路径配置（按你的本地路径修改）
# ============================================================
BENCHMARK_ROOT = r"D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark"

PYG_SPLIT_DIR = os.path.join(BENCHMARK_ROOT, "pyg_split_bundles")
SPLIT_DIR = os.path.join(BENCHMARK_ROOT, "splits")
SUMMARY_DIR = os.path.join(BENCHMARK_ROOT, "summary")

SPLIT_META_FILE = os.path.join(SUMMARY_DIR, "masked_seed_splits_meta.tsv")
SPLIT_LONG_FILE = os.path.join(SUMMARY_DIR, "masked_seed_splits_long.tsv")

OUT_DIR = os.path.join(BENCHMARK_ROOT, "deep_gloc_benchmark_results_all214_run3")
RUN_SUMMARY_DIR = os.path.join(OUT_DIR, "run_summary")
PRED_DIR = os.path.join(OUT_DIR, "predictions")
METRIC_DIR = os.path.join(OUT_DIR, "metrics")

for d in [OUT_DIR, RUN_SUMMARY_DIR, PRED_DIR, METRIC_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# 2. 训练配置
#    正式跑可用 N_RUNS=20；调试可先改成 3-5
# ============================================================
CONFIG = {
    "hidden_dim": 32,
    "heads_1": 4,
    "heads_2": 1,
    "dropout": 0.25,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "num_epochs": 250,
    "patience": 30,
    "lambda_orth": 1e-3,
    "use_seed_weight": True,
    "label_smoothing": 0.0
}

N_RUNS = 3
RUN_SEEDS = list(range(2026, 2026 + N_RUNS))

# ============================================================
# 3. 可选筛选
#    PROCESS_SPLIT_LIMITS = None 表示跑全部 214 个 split
#    输出目录已改为新文件夹，不会覆盖测试结果
# ============================================================
ONLY_PROCESSES = None
# 例如：ONLY_PROCESSES = ["Biosynthesis", "Excretion"]

#MAX_SPLITS_PER_PROCESS = None
# 例如：MAX_SPLITS_PER_PROCESS = 2  # 调试时每个过程只跑2个split
PROCESS_SPLIT_LIMITS = None
# 设为 None 表示跑全部 214 个 split；不会覆盖测试结果，因为 OUT_DIR 已改为新目录


In [3]:
# ============================================================
# 4. 基础函数
# ============================================================
def set_seed(seed: int = 2026):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def parse_split_id(x):
    return str(x).strip()

def load_split_meta():
    if os.path.exists(SPLIT_META_FILE):
        meta_df = pd.read_csv(SPLIT_META_FILE, sep="\t")
        meta_df["split_id"] = meta_df["split_id"].astype(str).map(parse_split_id)
        return meta_df

    # 没有 meta.tsv 时，直接扫描 pyg_split_bundles
    rows = []
    for p in sorted(Path(PYG_SPLIT_DIR).glob("*")):
        if p.is_dir():
            split_id = p.name
            process = split_id.split("__split_")[0] if "__split_" in split_id else split_id.split("_split_")[0]
            rows.append({
                "split_id": split_id,
                "process": process,
                "repeat_id": np.nan,
                "scheme": np.nan
            })
    return pd.DataFrame(rows)

split_meta_df = load_split_meta()
print("split_meta_df shape:", split_meta_df.shape)
display(split_meta_df.head())


split_meta_df shape: (214, 7)


,split_id,process,repeat_id,scheme,n_total,n_train,n_heldout
0,Biosynthesis__split_001,Biosynthesis,1,random_holdout,15,11,4
1,Biosynthesis__split_002,Biosynthesis,2,random_holdout,15,11,4
2,Biosynthesis__split_003,Biosynthesis,3,random_holdout,15,11,4
3,Biosynthesis__split_004,Biosynthesis,4,random_holdout,15,11,4
4,Biosynthesis__split_005,Biosynthesis,5,random_holdout,15,11,4


In [4]:
run_meta_df = split_meta_df.copy()

if ONLY_PROCESSES is not None:
    run_meta_df = run_meta_df[run_meta_df["process"].isin(ONLY_PROCESSES)].copy()

if PROCESS_SPLIT_LIMITS is not None:
    keep_parts = []
    for proc, sub in run_meta_df.groupby("process", group_keys=False):
        sub = sub.sort_values(["process", "repeat_id", "split_id"]).copy()
        limit = PROCESS_SPLIT_LIMITS.get(proc, None)
        if limit is None:
            keep_parts.append(sub)
        else:
            keep_parts.append(sub.head(limit))
    run_meta_df = pd.concat(keep_parts, axis=0).copy()

run_meta_df = run_meta_df.sort_values(["process", "repeat_id", "split_id"]).reset_index(drop=True)
print("Output directory:", OUT_DIR)
print("n_splits_to_run:", run_meta_df.shape[0])
display(run_meta_df.groupby("process")["split_id"].nunique().reset_index(name="n_splits"))


Output directory: D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\deep_gloc_benchmark_results_all214_run3
n_splits_to_run: 214


,process,n_splits
0,Biosynthesis,100
1,Esterification,6
2,Excretion,100
3,Uptake,8


In [5]:
# ============================================================
# 6. 模型定义（沿用你原来的 Deep_GLOC 主训练 notebook）
# ============================================================
class ExpansionGAT(nn.Module):
    def __init__(
        self,
        in_dim,
        edge_dim,
        hidden_dim,
        out_dim=4,
        heads_1=4,
        heads_2=1,
        dropout=0.25
    ):
        super().__init__()
        self.dropout = dropout

        self.gat1 = GATv2Conv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads_1,
            concat=True,
            dropout=dropout,
            edge_dim=edge_dim
        )

        self.gat2 = GATv2Conv(
            in_channels=hidden_dim * heads_1,
            out_channels=out_dim,
            heads=heads_2,
            concat=False,
            dropout=dropout,
            edge_dim=edge_dim
        )

    def forward(self, x, edge_index, edge_attr):
        x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        logits = self.gat2(x, edge_index, edge_attr)
        return logits


def orthogonality_loss(logits):
    probs = F.softmax(logits, dim=1)
    gram = probs.T @ probs
    eye = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - eye, p="fro")


def evaluate_supervised(model, data, label_smoothing=0.0):
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        pred = logits.argmax(dim=1)

        mask = data.supervised_mask
        if int(mask.sum()) == 0:
            return {"loss": np.nan, "acc": np.nan}

        loss = F.cross_entropy(
            logits[mask],
            data.y[mask],
            label_smoothing=label_smoothing
        ).item()

        acc = (pred[mask] == data.y[mask]).float().mean().item()

    return {"loss": loss, "acc": acc}


In [6]:
def train_one_run(data_cpu, process_map, config, run_seed, run_name="run"):
    set_seed(run_seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = copy.deepcopy(data_cpu).to(device)

    model = ExpansionGAT(
        in_dim=data.x.shape[1],
        edge_dim=data.edge_attr.shape[1],
        hidden_dim=config["hidden_dim"],
        out_dim=len(process_map),
        heads_1=config["heads_1"],
        heads_2=config["heads_2"],
        dropout=config["dropout"]
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_state = None
    best_monitor = np.inf
    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(1, config["num_epochs"] + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data.x, data.edge_index, data.edge_attr)
        mask = data.supervised_mask

        ce_vec = F.cross_entropy(
            logits[mask],
            data.y[mask],
            reduction="none",
            label_smoothing=config["label_smoothing"]
        )

        if config["use_seed_weight"]:
            ce = (ce_vec * data.seed_weight[mask]).mean()
        else:
            ce = ce_vec.mean()

        orth = orthogonality_loss(logits)
        loss = ce + config["lambda_orth"] * orth

        loss.backward()
        optimizer.step()

        monitor = loss.item()
        eval_res = evaluate_supervised(
            model,
            data,
            label_smoothing=config["label_smoothing"]
        )

        history.append({
            "epoch": epoch,
            "supervised_loss": eval_res["loss"],
            "supervised_acc": eval_res["acc"],
            "ce_loss": ce.item(),
            "orth_loss": orth.item(),
            "total_loss": loss.item()
        })

        if monitor < best_monitor:
            best_monitor = monitor
            best_epoch = epoch
            wait = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1

        if wait >= config["patience"]:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        prob = F.softmax(logits, dim=1).detach().cpu()
        pred = logits.argmax(dim=1).detach().cpu()

    return {
        "run_name": run_name,
        "seed": run_seed,
        "best_epoch": best_epoch,
        "best_monitor": best_monitor,
        "history": pd.DataFrame(history),
        "prob": prob,
        "pred": pred,
        "supervised_final": evaluate_supervised(
            model,
            data,
            label_smoothing=config["label_smoothing"]
        )
    }


In [7]:
# ============================================================
# 7. 读取单个 split 的 bundle
# ============================================================
def load_bundle_for_split(split_id):
    pt_file = os.path.join(PYG_SPLIT_DIR, split_id, "pyg_data_no_leak.pt")
    if not os.path.exists(pt_file):
        raise FileNotFoundError(f"pt 文件不存在: {pt_file}")

    bundle = torch.load(pt_file, map_location="cpu", weights_only=False)

    data = bundle["data"]
    node_df = bundle["node_df"].copy()
    aux_info = bundle.get("aux_info", {})
    benchmark_info = bundle.get("benchmark_info", {})

    # process_map
    if "process_map" in aux_info:
        process_map = aux_info["process_map"]
    else:
        process_map = {
            "Biosynthesis": 0,
            "Esterification": 1,
            "Excretion": 2,
            "Uptake": 3
        }

    label2process = {v: k for k, v in process_map.items()}
    process_names = [label2process[i] for i in sorted(label2process)]

    # idx2gene / gene2idx
    if "idx2gene" in aux_info:
        idx2gene = aux_info["idx2gene"]
    else:
        idx2gene = {i: str(g) for i, g in enumerate(node_df["ENSGID"].tolist())}

    if "gene2idx" in aux_info:
        gene2idx = aux_info["gene2idx"]
    else:
        gene2idx = {g: i for i, g in idx2gene.items()}

    # edge_attr
    if not hasattr(data, "edge_attr") or data.edge_attr is None:
        if hasattr(data, "edge_weight"):
            data.edge_attr = data.edge_weight.view(-1, 1).float()
        else:
            data.edge_attr = torch.ones(data.edge_index.shape[1], 1, dtype=torch.float32)

    # supervised mask
    if not hasattr(data, "supervised_mask"):
        if hasattr(data, "train_mask"):
            data.supervised_mask = data.train_mask.clone()
        else:
            data.supervised_mask = (data.y >= 0)

    # all_core_mask：尽量保留原始属性
    if not hasattr(data, "all_core_mask"):
        data.all_core_mask = (data.y >= 0)

    # seed weight
    if not hasattr(data, "seed_weight"):
        data.seed_weight = torch.ones(data.num_nodes, dtype=torch.float32)

    if "ENSGID" not in node_df.columns:
        node_df["ENSGID"] = [idx2gene[i] for i in range(len(node_df))]

    # held-out genes
    heldout_path = os.path.join(SPLIT_DIR, split_id, "heldout_seeds.tsv")
    if os.path.exists(heldout_path):
        held_df = pd.read_csv(heldout_path, sep="\t")
        heldout_genes = held_df.iloc[:, 0].astype(str).str.strip().str.split(".").str[0].tolist()
    else:
        heldout_genes = benchmark_info.get("heldout_genes", [])

    return {
        "bundle": bundle,
        "data": data,
        "node_df": node_df,
        "process_map": process_map,
        "label2process": label2process,
        "process_names": process_names,
        "idx2gene": idx2gene,
        "gene2idx": gene2idx,
        "heldout_genes": heldout_genes,
        "benchmark_info": benchmark_info,
        "pt_file": pt_file
    }


In [8]:
# ============================================================
# 8. 预测聚合与指标计算
# ============================================================
def aggregate_predictions(all_results):
    prob_stack = torch.stack([res["prob"] for res in all_results], dim=0)   # [R, N, C]
    pred_stack = torch.stack([res["pred"] for res in all_results], dim=0)   # [R, N]

    mean_prob = prob_stack.mean(dim=0)
    std_prob = prob_stack.std(dim=0)

    mean_pred = mean_prob.argmax(dim=1)
    max_prob, _ = mean_prob.max(dim=1)

    sorted_prob, _ = torch.sort(mean_prob, dim=1, descending=True)
    second_prob = sorted_prob[:, 1]
    margin = max_prob - second_prob

    pred_freq = []
    for i in range(mean_prob.shape[0]):
        cls = int(mean_pred[i].item())
        freq = (pred_stack[:, i] == cls).float().mean().item()
        pred_freq.append(freq)
    pred_freq = torch.tensor(pred_freq, dtype=torch.float32)

    return {
        "prob_stack": prob_stack,
        "pred_stack": pred_stack,
        "mean_prob": mean_prob,
        "std_prob": std_prob,
        "mean_pred": mean_pred,
        "max_prob": max_prob,
        "second_prob": second_prob,
        "margin": margin,
        "pred_freq": pred_freq
    }


def build_prediction_table(split_id, process_name, data, idx2gene, label2process, agg):
    pred_df = pd.DataFrame({
        "split_id": split_id,
        "ENSGID": [idx2gene[i] for i in range(data.num_nodes)],
        "pred_label": agg["mean_pred"].numpy(),
        "pred_process": [label2process[int(x)] for x in agg["mean_pred"].numpy()],
        "mean_max_prob": agg["max_prob"].numpy(),
        "mean_second_prob": agg["second_prob"].numpy(),
        "mean_margin": agg["margin"].numpy(),
        "pred_freq": agg["pred_freq"].numpy(),
        "is_supervised_seed": data.supervised_mask.cpu().numpy().astype(int),
        "is_all_core": data.all_core_mask.cpu().numpy().astype(int),
    })

    for lab, proc in label2process.items():
        pred_df[f"prob_{proc}"] = agg["mean_prob"][:, lab].numpy()
        pred_df[f"prob_sd_{proc}"] = agg["std_prob"][:, lab].numpy()

    # 当前 split 评估用 score
    pred_df["eval_process"] = process_name
    pred_df["eval_score"] = pred_df[f"prob_{process_name}"]

    # 候选集合：不在 supervised seeds 中
    pred_df["is_candidate"] = (pred_df["is_supervised_seed"] == 0).astype(int)

    # 在候选集合内部排序，1 为最高
    cand_df = pred_df[pred_df["is_candidate"] == 1].copy()
    cand_df = cand_df.sort_values(
        ["eval_score", "mean_margin", "pred_freq", "ENSGID"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)
    cand_df["candidate_rank"] = np.arange(1, cand_df.shape[0] + 1)

    pred_df = pred_df.merge(
        cand_df[["ENSGID", "candidate_rank"]],
        on="ENSGID",
        how="left"
    )

    return pred_df


def compute_metrics_from_pred_df(pred_df, heldout_genes):
    heldout_genes = [str(g).split(".")[0] for g in heldout_genes]
    heldout_genes = sorted(set(heldout_genes))

    cand_df = pred_df[pred_df["is_candidate"] == 1].copy()
    cand_df["is_positive"] = cand_df["ENSGID"].isin(heldout_genes).astype(int)

    n_candidates = cand_df.shape[0]
    n_pos = int(cand_df["is_positive"].sum())
    n_neg = n_candidates - n_pos

    held_ranks = (
        cand_df.loc[cand_df["is_positive"] == 1, "candidate_rank"]
        .dropna()
        .astype(int)
        .tolist()
    )

    def recall_at_k(ranks, k):
        if len(ranks) == 0:
            return np.nan
        return np.mean([r <= k for r in ranks])

    metrics = {
        "n_candidates": n_candidates,
        "n_heldout": n_pos,
        "n_negatives": n_neg,
        "Recall@20": recall_at_k(held_ranks, 20),
        "Recall@50": recall_at_k(held_ranks, 50),
        "Recall@100": recall_at_k(held_ranks, 100),
        "median_rank": float(np.median(held_ranks)) if len(held_ranks) > 0 else np.nan,
        "MRR": float(np.mean([1.0 / r for r in held_ranks])) if len(held_ranks) > 0 else np.nan,
    }

    if n_pos > 0 and n_neg > 0:
        y_true = cand_df["is_positive"].values
        y_score = cand_df["eval_score"].values
        metrics["AUROC"] = roc_auc_score(y_true, y_score)
        metrics["AUPRC"] = average_precision_score(y_true, y_score)
    else:
        metrics["AUROC"] = np.nan
        metrics["AUPRC"] = np.nan

    # 单独输出 held-out rank 表，方便排查
    held_rank_df = cand_df[cand_df["is_positive"] == 1][
        ["ENSGID", "candidate_rank", "eval_score"]
    ].sort_values("candidate_rank").reset_index(drop=True)

    return metrics, held_rank_df


In [9]:
# ============================================================
# 9. 单个 split 跑 Deep_GLOC benchmark
# ============================================================
def run_one_split(split_id, process_name):
    print("=" * 100)
    print("Running split:", split_id, "| process:", process_name)
    print("=" * 100)

    split_start_time = time.time()

    obj = load_bundle_for_split(split_id)
    data = obj["data"]
    process_map = obj["process_map"]
    label2process = obj["label2process"]
    idx2gene = obj["idx2gene"]
    heldout_genes = obj["heldout_genes"]

    print("supervised seeds:", int(data.supervised_mask.sum().item()))
    print("heldout genes:", len(heldout_genes))

    all_results = []
    run_time_rows = []

    for i, run_seed in enumerate(RUN_SEEDS, start=1):
        run_name = f"{split_id}__run_{i:02d}_seed_{run_seed}"
        print(f"  [{i}/{len(RUN_SEEDS)}] {run_name}")

        run_start_time = time.time()

        res = train_one_run(
            data_cpu=data,
            process_map=process_map,
            config=CONFIG,
            run_seed=run_seed,
            run_name=run_name
        )

        run_elapsed = time.time() - run_start_time
        print(f"      finished in {run_elapsed/60:.2f} min ({run_elapsed:.1f} sec)")

        run_time_rows.append({
            "split_id": split_id,
            "process": process_name,
            "run_name": run_name,
            "seed": run_seed,
            "elapsed_sec": run_elapsed,
            "elapsed_min": run_elapsed / 60
        })

        all_results.append(res)

    split_elapsed = time.time() - split_start_time
    print(f"Split {split_id} finished in {split_elapsed/60:.2f} min ({split_elapsed:.1f} sec)")

    # 训练摘要
    train_summary_df = pd.DataFrame([{
        "split_id": split_id,
        "process": process_name,
        "run_name": res["run_name"],
        "seed": res["seed"],
        "best_epoch": res["best_epoch"],
        "best_monitor": res["best_monitor"],
        "supervised_loss": res["supervised_final"]["loss"],
        "supervised_acc": res["supervised_final"]["acc"]
    } for res in all_results])

    # 合并每次 run 的耗时
    run_time_df = pd.DataFrame(run_time_rows)
    train_summary_df = train_summary_df.merge(
        run_time_df,
        on=["split_id", "process", "run_name", "seed"],
        how="left"
    )

    train_summary_file = os.path.join(RUN_SUMMARY_DIR, f"{split_id}__training_summary.tsv")
    train_summary_df.to_csv(train_summary_file, sep="\t", index=False)

    agg = aggregate_predictions(all_results)

    pred_df = build_prediction_table(
        split_id=split_id,
        process_name=process_name,
        data=data,
        idx2gene=idx2gene,
        label2process=label2process,
        agg=agg
    )

    metrics, held_rank_df = compute_metrics_from_pred_df(pred_df, heldout_genes)
    metrics_row = {
        "split_id": split_id,
        "process": process_name,
        "split_elapsed_sec": split_elapsed,
        "split_elapsed_min": split_elapsed / 60,
        **metrics
    }

    pred_file = os.path.join(PRED_DIR, f"{split_id}__ranked_predictions.tsv")
    pred_df.to_csv(pred_file, sep="\t", index=False)

    held_rank_file = os.path.join(PRED_DIR, f"{split_id}__heldout_ranks.tsv")
    held_rank_df.to_csv(held_rank_file, sep="\t", index=False)

    return {
        "split_id": split_id,
        "process": process_name,
        "pred_df": pred_df,
        "train_summary_df": train_summary_df,
        "metrics_row": metrics_row,
        "held_rank_df": held_rank_df,
        "pred_file": pred_file,
        "held_rank_file": held_rank_file,
        "train_summary_file": train_summary_file
    }

In [10]:
# ============================================================
# 10. 批量运行全部 split
# ============================================================
all_metric_rows = []
all_held_rank_rows = []

for _, row in run_meta_df.iterrows():
    split_id = str(row["split_id"])
    process_name = str(row["process"])

    res = run_one_split(split_id, process_name)

    all_metric_rows.append(res["metrics_row"])

    tmp = res["held_rank_df"].copy()
    tmp["split_id"] = split_id
    tmp["process"] = process_name
    all_held_rank_rows.append(tmp)

split_metrics_df = pd.DataFrame(all_metric_rows)
held_rank_all_df = pd.concat(all_held_rank_rows, axis=0, ignore_index=True) if len(all_held_rank_rows) > 0 else pd.DataFrame()

split_metrics_file = os.path.join(METRIC_DIR, "deep_gloc_split_metrics.tsv")
held_rank_all_file = os.path.join(METRIC_DIR, "deep_gloc_all_heldout_ranks.tsv")

split_metrics_df.to_csv(split_metrics_file, sep="\t", index=False)
held_rank_all_df.to_csv(held_rank_all_file, sep="\t", index=False)

print("saved:")
print(split_metrics_file)
print(held_rank_all_file)

display(split_metrics_df.head())


Running split: Biosynthesis__split_001 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_001__run_01_seed_2026
      finished in 3.47 min (208.1 sec)
  [2/3] Biosynthesis__split_001__run_02_seed_2027
      finished in 4.46 min (267.9 sec)
  [3/3] Biosynthesis__split_001__run_03_seed_2028
      finished in 6.75 min (405.0 sec)
Split Biosynthesis__split_001 finished in 14.69 min (881.7 sec)
Running split: Biosynthesis__split_002 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_002__run_01_seed_2026
      finished in 3.46 min (207.7 sec)
  [2/3] Biosynthesis__split_002__run_02_seed_2027
      finished in 5.92 min (355.0 sec)
  [3/3] Biosynthesis__split_002__run_03_seed_2028
      finished in 6.49 min (389.4 sec)
Split Biosynthesis__split_002 finished in 15.88 min (952.7 sec)
Running split: Biosynthesis__split_003 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_003__ru

Running split: Biosynthesis__split_014 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_014__run_01_seed_2026
      finished in 3.30 min (198.1 sec)
  [2/3] Biosynthesis__split_014__run_02_seed_2027
      finished in 3.33 min (199.9 sec)
  [3/3] Biosynthesis__split_014__run_03_seed_2028
      finished in 4.66 min (279.8 sec)
Split Biosynthesis__split_014 finished in 11.31 min (678.4 sec)
Running split: Biosynthesis__split_015 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_015__run_01_seed_2026
      finished in 3.31 min (198.9 sec)
  [2/3] Biosynthesis__split_015__run_02_seed_2027
      finished in 4.36 min (261.6 sec)
  [3/3] Biosynthesis__split_015__run_03_seed_2028
      finished in 4.77 min (285.9 sec)
Split Biosynthesis__split_015 finished in 12.45 min (746.9 sec)
Running split: Biosynthesis__split_016 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_016__ru

Running split: Biosynthesis__split_027 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_027__run_01_seed_2026
      finished in 3.29 min (197.1 sec)
  [2/3] Biosynthesis__split_027__run_02_seed_2027
      finished in 7.18 min (430.8 sec)
  [3/3] Biosynthesis__split_027__run_03_seed_2028
      finished in 4.65 min (278.8 sec)
Split Biosynthesis__split_027 finished in 15.12 min (907.4 sec)
Running split: Biosynthesis__split_028 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_028__run_01_seed_2026
      finished in 7.20 min (432.0 sec)
  [2/3] Biosynthesis__split_028__run_02_seed_2027
      finished in 3.29 min (197.3 sec)
  [3/3] Biosynthesis__split_028__run_03_seed_2028
      finished in 4.59 min (275.5 sec)
Split Biosynthesis__split_028 finished in 15.09 min (905.5 sec)
Running split: Biosynthesis__split_029 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_029__ru

Running split: Biosynthesis__split_040 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_040__run_01_seed_2026
      finished in 4.62 min (277.4 sec)
  [2/3] Biosynthesis__split_040__run_02_seed_2027
      finished in 6.38 min (382.7 sec)
  [3/3] Biosynthesis__split_040__run_03_seed_2028
      finished in 4.41 min (264.3 sec)
Split Biosynthesis__split_040 finished in 15.42 min (925.1 sec)
Running split: Biosynthesis__split_041 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_041__run_01_seed_2026
      finished in 7.19 min (431.3 sec)
  [2/3] Biosynthesis__split_041__run_02_seed_2027
      finished in 4.28 min (256.6 sec)
  [3/3] Biosynthesis__split_041__run_03_seed_2028
      finished in 4.60 min (275.7 sec)
Split Biosynthesis__split_041 finished in 16.07 min (964.3 sec)
Running split: Biosynthesis__split_042 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_042__ru

Running split: Biosynthesis__split_053 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_053__run_01_seed_2026
      finished in 3.30 min (198.2 sec)
  [2/3] Biosynthesis__split_053__run_02_seed_2027
      finished in 4.29 min (257.5 sec)
  [3/3] Biosynthesis__split_053__run_03_seed_2028
      finished in 4.39 min (263.7 sec)
Split Biosynthesis__split_053 finished in 12.00 min (719.9 sec)
Running split: Biosynthesis__split_054 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_054__run_01_seed_2026
      finished in 7.19 min (431.5 sec)
  [2/3] Biosynthesis__split_054__run_02_seed_2027
      finished in 3.27 min (196.3 sec)
  [3/3] Biosynthesis__split_054__run_03_seed_2028
      finished in 4.62 min (276.9 sec)
Split Biosynthesis__split_054 finished in 15.09 min (905.4 sec)
Running split: Biosynthesis__split_055 | process: Biosynthesis
supervised seeds: 35
heldout genes: 4
  [1/3] Biosynthesis__split_055__ru

...

supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_066__run_01_seed_2026
      finished in 2.97 min (178.0 sec)
  [2/3] Excretion__split_066__run_02_seed_2027
      finished in 1.36 min (81.5 sec)
  [3/3] Excretion__split_066__run_03_seed_2028
      finished in 2.62 min (157.3 sec)
Split Excretion__split_066 finished in 6.95 min (416.9 sec)
Running split: Excretion__split_067 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_067__run_01_seed_2026
      finished in 2.80 min (168.2 sec)
  [2/3] Excretion__split_067__run_02_seed_2027
      finished in 1.36 min (81.4 sec)
  [3/3] Excretion__split_067__run_03_seed_2028
      finished in 1.90 min (114.2 sec)
Split Excretion__split_067 finished in 6.07 min (364.0 sec)
Running split: Excretion__split_068 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_068__run_01_seed_2026
      finished in 1.93 min (115.9 sec)
  [2/3] Excretion__split_068__run_02_seed_2027
     

      finished in 1.90 min (114.1 sec)
Split Excretion__split_079 finished in 7.83 min (469.8 sec)
Running split: Excretion__split_080 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_080__run_01_seed_2026
      finished in 2.29 min (137.2 sec)
  [2/3] Excretion__split_080__run_02_seed_2027
      finished in 2.96 min (177.6 sec)
  [3/3] Excretion__split_080__run_03_seed_2028
      finished in 1.90 min (113.7 sec)
Split Excretion__split_080 finished in 7.14 min (428.6 sec)
Running split: Excretion__split_081 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_081__run_01_seed_2026
      finished in 2.96 min (177.8 sec)
  [2/3] Excretion__split_081__run_02_seed_2027
      finished in 2.96 min (177.7 sec)
  [3/3] Excretion__split_081__run_03_seed_2028
      finished in 1.91 min (114.4 sec)
Split Excretion__split_081 finished in 7.83 min (470.0 sec)
Running split: Excretion__split_082 | process: Excretion
supervised seeds: 

supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_093__run_01_seed_2026
      finished in 2.29 min (137.2 sec)
  [2/3] Excretion__split_093__run_02_seed_2027
      finished in 2.96 min (177.8 sec)
  [3/3] Excretion__split_093__run_03_seed_2028
      finished in 1.92 min (115.5 sec)
Split Excretion__split_093 finished in 7.18 min (430.8 sec)
Running split: Excretion__split_094 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_094__run_01_seed_2026
      finished in 2.97 min (178.0 sec)
  [2/3] Excretion__split_094__run_02_seed_2027
      finished in 2.96 min (177.5 sec)
  [3/3] Excretion__split_094__run_03_seed_2028
      finished in 2.62 min (157.2 sec)
Split Excretion__split_094 finished in 8.55 min (512.9 sec)
Running split: Excretion__split_095 | process: Excretion
supervised seeds: 36
heldout genes: 3
  [1/3] Excretion__split_095__run_01_seed_2026
      finished in 2.40 min (144.0 sec)
  [2/3] Excretion__split_095__run_02_seed_2027
   

supervised seeds: 38
heldout genes: 1
  [1/3] Uptake__split_007__run_01_seed_2026
      finished in 2.79 min (167.6 sec)
  [2/3] Uptake__split_007__run_02_seed_2027
      finished in 2.96 min (177.8 sec)
  [3/3] Uptake__split_007__run_03_seed_2028
      finished in 1.89 min (113.6 sec)
Split Uptake__split_007 finished in 7.65 min (459.2 sec)
Running split: Uptake__split_008 | process: Uptake
supervised seeds: 38
heldout genes: 1
  [1/3] Uptake__split_008__run_01_seed_2026
      finished in 2.80 min (167.9 sec)
  [2/3] Uptake__split_008__run_02_seed_2027
      finished in 2.97 min (178.0 sec)
  [3/3] Uptake__split_008__run_03_seed_2028
      finished in 1.90 min (114.2 sec)
Split Uptake__split_008 finished in 7.67 min (460.2 sec)
saved:
D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\deep_gloc_benchmark_results_all214_run3\metrics\deep_gloc_split_metrics.tsv
D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\deep_gloc_benchm

,split_id,process,split_elapsed_sec,split_elapsed_min,n_candidates,n_heldout,n_negatives,Recall@20,Recall@50,Recall@100,median_rank,MRR,AUROC,AUPRC
0,Biosynthesis__split_001,Biosynthesis,881.666543,14.694442,3176,4,3172,0.25,1.00,1.0,37.0,0.034860,0.990227,0.071869
1,Biosynthesis__split_002,Biosynthesis,952.712492,15.878542,3176,4,3172,0.75,1.00,1.0,8.5,0.362060,0.997557,0.513716
2,Biosynthesis__split_003,Biosynthesis,893.568536,14.892809,3176,4,3172,0.50,0.75,1.0,23.5,0.086263,0.989124,0.142372
3,Biosynthesis__split_004,Biosynthesis,732.286226,12.204770,3176,4,3172,0.25,0.75,1.0,38.5,0.037429,0.989360,0.070037
4,Biosynthesis__split_005,Biosynthesis,1089.363759,18.156063,3176,4,3172,0.50,0.75,1.0,28.0,0.111772,0.991882,0.155423


In [11]:
# ============================================================
# 11. 按 process 汇总
# ============================================================
metric_cols = ["Recall@20", "Recall@50", "Recall@100", "median_rank", "MRR", "AUROC", "AUPRC"]

process_summary_df = (
    split_metrics_df.groupby("process")[metric_cols]
    .agg(["mean", "std", "median", "count"])
)

# 拉平列名
process_summary_df.columns = [f"{a}__{b}" for a, b in process_summary_df.columns]
process_summary_df = process_summary_df.reset_index()

process_summary_file = os.path.join(METRIC_DIR, "deep_gloc_process_summary.tsv")
process_summary_df.to_csv(process_summary_file, sep="\t", index=False)

print("saved:", process_summary_file)
display(process_summary_df)


saved: D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\deep_gloc_benchmark_results_all214_run3\metrics\deep_gloc_process_summary.tsv


,process,Recall@20__mean,Recall@20__std,Recall@20__median,Recall@20__count,Recall@50__mean,Recall@50__std,Recall@50__median,Recall@50__count,Recall@100__mean,...,MRR__median,MRR__count,AUROC__mean,AUROC__std,AUROC__median,AUROC__count,AUPRC__mean,AUPRC__std,AUPRC__median,AUPRC__count
0,Biosynthesis,0.632500,0.247347,0.750000,100,0.885000,0.156589,1.000000,100,0.967500,...,0.132409,100,0.986854,0.052479,0.994168,100,0.274465,0.201639,0.206429,100
1,Esterification,0.500000,0.547723,0.500000,6,0.833333,0.408248,1.000000,6,1.000000,...,0.177305,6,0.991173,0.009873,0.992434,6,0.398119,0.481907,0.177305,6
2,Excretion,0.716667,0.269576,0.666667,100,0.753333,0.239809,0.666667,100,0.793333,...,0.246096,100,0.899158,0.125177,0.963377,100,0.376207,0.216420,0.361111,100
3,Uptake,0.750000,0.462910,1.000000,8,0.750000,0.462910,1.000000,8,0.750000,...,1.000000,8,0.840558,0.351861,1.000000,8,0.750180,0.462576,1.000000,8


In [12]:
# ============================================================
# 12. macro-average（四个 process 等权）
# ============================================================
# 这里不是把所有 split 混在一起平均，而是先按 process 求均值，再对 process 做平均
process_means_df = (
    split_metrics_df.groupby("process")[metric_cols]
    .mean()
    .reset_index()
)

macro_row = {"level": "macro_average"}
for c in metric_cols:
    macro_row[c] = process_means_df[c].mean()

macro_df = pd.DataFrame([macro_row])

macro_file = os.path.join(METRIC_DIR, "deep_gloc_macro_average.tsv")
macro_df.to_csv(macro_file, sep="\t", index=False)

print("saved:", macro_file)
display(macro_df)


saved: D:\phd cases\metabolic analysis\code\Deep_GLOC\run_GAT_new\masked_seed_benchmark\deep_gloc_benchmark_results_all214_run3\metrics\deep_gloc_macro_average.tsv


,level,Recall@20,Recall@50,Recall@100,median_rank,MRR,AUROC,AUPRC
0,macro_average,0.649792,0.805417,0.877708,151.12125,0.39852,0.929436,0.449743


## 结果解释建议

### 1. 每个 split 的指标
`deep_gloc_split_metrics.tsv`  
这是最底层 benchmark 结果。

### 2. 每个 process 的平均结果
`deep_gloc_process_summary.tsv`  
建议用于和 RWR / DeepWalk 做 process 内比较。

### 3. macro-average
`deep_gloc_macro_average.tsv`  
建议作为四个过程整体表现的主结果，因为它不会让 split 多的过程权重更大。

## 和师弟结果合并时
后续你只需要把师弟的 RWR / DeepWalk 结果也整理成同样的 split-level metrics 表，再按：
- `process`
- `split_id`
- `method`

合并即可。


###保存环境数据
import os
import dill
import json
import types
from datetime import datetime

save_dir = f"session_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(save_dir, exist_ok=True)

skip_names = {
    "In", "Out", "exit", "quit", "get_ipython"
}

saved = {}
skipped = {}

for k, v in list(globals().items()):
    if k.startswith("_") or k in skip_names:
        continue
    if isinstance(v, (
        types.ModuleType,
        types.FunctionType,
        types.BuiltinFunctionType,
        type
    )):
        skipped[k] = "module/function/class"
        continue

    try:
        dill.dumps(v)
        saved[k] = v
    except Exception as e:
        skipped[k] = str(e)

workspace_file = os.path.join(save_dir, "workspace.pkl")
with open(workspace_file, "wb") as f:
    dill.dump(saved, f)

with open(os.path.join(save_dir, "saved_names.json"), "w", encoding="utf-8") as f:
    json.dump(sorted(saved.keys()), f, ensure_ascii=False, indent=2)

with open(os.path.join(save_dir, "skipped_names.json"), "w", encoding="utf-8") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print(f"保存完成: {workspace_file}")
print(f"成功保存变量数: {len(saved)}")
print(f"跳过变量数: {len(skipped)}")
print("已保存变量示例:", list(saved.keys())[:20])

import json
with open("D:/phd cases/metabolic analysis/code/Deep_GLOC/run_GAT_new/masked_seed_benchmark/deep_gloc_benchmark_results_all214_run3/session_backup_20260322_134348/saved_names.json", "r", encoding="utf-8") as f:
    saved_names = json.load(f)

print(saved_names)

print(type(res))
print(res.keys() if isinstance(res, dict) else "res is not a dict")